# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, examine, and process the FAIR² dataset on extracellular vesicles from human mast cells using the `mlcroissant` library. The dataset contains protein analysis derived from mass spectrometry, including abundance, sequence, and modification information, and is structured via the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review all available record sets, their `@id`s, and the fields within them. This overview assists in selecting what to load for analysis.

**Entities are referenced by their `@id` field as per the dataset schema.**

In [ ]:
# List all record sets and the fields/columns within each
record_sets = list(dataset.list_record_sets())  # returns list of dicts of each record set's metadata
print('Available record sets and their fields:')
for rs in record_sets:
    rs_id = rs['@id']
    rs_name = rs.get('name', '<unnamed>')
    print(f'  RecordSet @id: {rs_id}')
    print(f'       name : {rs_name}')
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        f_id = f.get('@id', str(f))
        fname = f.get('name', f_id)
        print(f'         Field @id: {f_id} name: {fname}')
    print('-'*65)

## 3. Data Extraction
Select specific record set(s) for loading. This section demonstrates loading all records from a main record set into a pandas DataFrame using only the `@id` of each record set and field for reference. 

+ **Set the `record_sets_to_load` variable to contain the `@id`s of the record sets you wish to extract.**
+ **The IDs and field names are from the schema.**

In [ ]:
#--- Define your record sets here by the @id as shown above ---#
record_sets_to_load = [
    'https://api.app.sen.science/frontiers/7154140/9ab71f93-9fc6-4f6a-93c1-294ee97b34ac'  # Replace/example: main proteins table; adjust if more record sets
]

dataframes = {}
for rs_id in record_sets_to_load:
    records = list(dataset.records(record_set=rs_id))
    if len(records) == 0:
        print(f'No records found for: {rs_id}')
        continue
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f'Loaded {len(df)} records for record set {rs_id}\nColumns:')
    print(list(df.columns))
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's process and explore the main protein abundance table.

- **All fields are referenced via their `@id`.**
- We'll filter for proteins with molecular weight (MW) above a threshold, normalize their abundance, and group by a categorical variable such as 'sample' or similar, as available.
- Adjust the field `@id` values to match the schema for the actual field IDs.


In [ ]:
# Parameters: update these with the correct field @id from the field overview
main_rs_id = record_sets_to_load[0]
df = dataframes[main_rs_id]

# Example field IDs, adjust if the IDs are different (inspect columns from previous cell)
# e.g., '@id': 'https://api.app.sen.science/frontiers/7154140/3e9a2e16-d236-4e0e-8b23-a609d2038a47'
mw_id = 'https://api.app.sen.science/frontiers/7154140/3e9a2e16-d236-4e0e-8b23-a609d2038a47'  # Molecular Weight
abundance_id = 'https://api.app.sen.science/frontiers/7154140/80034d93-b69e-4675-9838-6f87c9e0a7d6'  # Normalized Abundance
group_id = 'https://api.app.sen.science/frontiers/7154140/ebcc3635-ac0c-4b9b-9566-bb81f5ff3fb2'   # Sample Type or Label (edit to match available categorical field)

# Filter for MW > threshold
threshold = 60000  # e.g., 60 kDa
if mw_id in df.columns:
    filtered_df = df[df[mw_id] > threshold].copy()
    print(f'Filtered records with molecular weight ({mw_id}) > {threshold}:')
    display(filtered_df[[mw_id, abundance_id]].head())
    
    # Normalize protein abundance
    if abundance_id in filtered_df.columns:
        norm_col = f'{abundance_id}_normalized'
        filtered_df[norm_col] = (filtered_df[abundance_id] - filtered_df[abundance_id].mean())/filtered_df[abundance_id].std()
        print('Normalized abundance:')
        display(filtered_df[[abundance_id, norm_col]].head())
    else:
        print(f'Abundance field {abundance_id} not found in columns.')
else:
    print(f'Molecular Weight field {mw_id} not found in this table.')

# Optional: grouping by sample type
if group_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_id).mean(numeric_only=True)
    print(f'Grouped mean values by {group_id}:')
    display(grouped_df.head())
else:
    print(f'Grouping field {group_id} not found in this table.')

## 5. Visualization
Visualize the abundance distribution for proteins with high molecular weight. We'll plot a histogram of normalized abundance and a scatter plot of MW vs. abundance.

- All field references are made by their field `@id`.
- Plots only if the respective columns exist in the DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if mw_id in filtered_df.columns and abundance_id in filtered_df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[abundance_id], bins=30, kde=True)
    plt.xlabel('Normalized Abundance')
    plt.title('Distribution of Protein Abundance (MW > 60kDa)')
    plt.show()
    
    # Scatter plot
    plt.figure(figsize=(7, 6))
    plt.scatter(filtered_df[mw_id], filtered_df[abundance_id], alpha=0.7)
    plt.xlabel('Molecular Weight (@id)')
    plt.ylabel('Normalized Abundance (@id)')
    plt.title('Protein Molecular Weight vs. Abundance')
    plt.show()

## 6. Conclusion
- We loaded and explored the FAIR² mass spectrometry dataset via Croissant schema and `mlcroissant`.
- Used only `@id` references for record sets and fields, as recommended.
- Extracted and processed protein abundance and molecular weight, identified patterns and visualized distributions.
- The dataset can support biomarker discovery, comparative protein studies, and further ML or bioinformatic workflows.

**Adjust field `@id` variables if your actual dataset has different IDs or you want to analyze other features or record sets.**